# Test4 Phase 1 — GTVp / GTVn preprocessing preview

Run one **RADCURE** and one **HECKTOR** case with `tumor_label_mode="separate"` and inspect organ + tumour overlays.

**Requirements:** GPU (TotalSegmentator), `.env` with AWS keys (RADCURE), HECKTOR NIfTI on disk.

If a case was already processed in merged mode, delete its `output/` folder first (HECKTOR) or use a fresh work dir.

## 1. Setup

In [ ]:
import os
import sys
import shutil
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Repo root (adjust if notebook is copied elsewhere)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "image_processor").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent  # from research_notebooks/test4_gtvp_gtvn/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from image_processor import (
    CaseProcessor,
    RADCURE,
    HECKTOR,
    TUMOR_LABEL_MODE_SEPARATE,
)
from image_processor.visualization.visualizer import MedicalImageVisualizer
import nibabel as nib
import numpy as np

TUMOR_LABEL_MODE = TUMOR_LABEL_MODE_SEPARATE
NOTEBOOK_DIR = REPO_ROOT / "research_notebooks" / "test4_gtvp_gtvn"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
ORGAN_DICT_PATH = NOTEBOOK_DIR / "radcure_dictionary_test4.json"

# --- configure cases (edit for your server) ---
RADCURE_CASE_ID = os.getenv("TEST4_RADCURE_CASE", "RADCURE-0005")
HECKTOR_CASE_ID = os.getenv("TEST4_HECKTOR_CASE", "CHUM-001")

MAIN_PATH = os.getenv("MAIN_PATH", str(REPO_ROOT / "work" / "test4_preview"))
HECKTOR_CASES_ROOT = os.getenv("HECKTOR_CASES_ROOT", "/path/to/hecktor/cases")

AWS_BUCKET = os.getenv("AWS_BUCKET_NAME", "your-bucket")
AWS_FOLDER = os.getenv("AWS_FOLDER", "RADCURE/all_cases/")

CUDA_DEVICE = os.getenv("CUDA_VISIBLE_DEVICES", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE

print("Repo:", REPO_ROOT)
print("Tumor label mode:", TUMOR_LABEL_MODE)
print("Organ dictionary:", ORGAN_DICT_PATH)
print("RADCURE case:", RADCURE_CASE_ID)
print("HECKTOR case:", HECKTOR_CASE_ID, "root:", HECKTOR_CASES_ROOT)

## 2. Process one HECKTOR case

In [ ]:
hecktor_case_dir = Path(HECKTOR_CASES_ROOT) / HECKTOR_CASE_ID
hecktor_output = hecktor_case_dir / "output"

# Remove old merged-mode outputs so CaseProcessor re-runs
if hecktor_output.is_dir():
    print("Removing existing output/ for fresh Test4 run:", hecktor_output)
    shutil.rmtree(hecktor_output)

processor_hecktor = CaseProcessor(
    main_path=MAIN_PATH,
    aws_bucket_name="dummy",
    aws_folder="dummy/",
    organ_dictionary_path=str(ORGAN_DICT_PATH),
    convention=HECKTOR,
    cases_root=HECKTOR_CASES_ROOT,
    tumor_label_mode=TUMOR_LABEL_MODE,
    hecktor_cleanup_intermediates=False,
)

hecktor_result = processor_hecktor.process_case(HECKTOR_CASE_ID)
print(hecktor_result)

In [ ]:
# Quick label check — expect both GTVp and GTVn indices in combined mask
hecktor_labels = nib.load(hecktor_result["label_path"]).get_fdata().astype(int)
organ_dict = processor_hecktor.organ_dictionary.copy()
inv = {v: k for k, v in organ_dict.items()}
unique = sorted(set(np.unique(hecktor_labels)) - {0})
print("Unique label indices:", unique)
print("Tumors in dictionary:", {k: organ_dict.get(k) for k in ("GTVp", "GTVn")})
print("Labels present:", [inv.get(i, f"idx_{i}") for i in unique if i in inv])

## 3. Process one RADCURE case

In [ ]:
# Fresh dictionary for RADCURE run (organs may differ slightly per case)
radcure_dict_path = NOTEBOOK_DIR / "radcure_dictionary_test4_radcure.json"
if radcure_dict_path.exists():
    radcure_dict_path.unlink()

processor_radcure = CaseProcessor(
    main_path=MAIN_PATH,
    aws_bucket_name=AWS_BUCKET,
    aws_folder=AWS_FOLDER,
    organ_dictionary_path=str(radcure_dict_path),
    convention=RADCURE,
    tumor_label_mode=TUMOR_LABEL_MODE,
)

radcure_result = processor_radcure.process_case(RADCURE_CASE_ID)
print(radcure_result)

In [ ]:
radcure_labels = nib.load(radcure_result["label_path"]).get_fdata().astype(int)
organ_dict_r = processor_radcure.organ_dictionary.copy()
inv_r = {v: k for k, v in organ_dict_r.items()}
unique_r = sorted(set(np.unique(radcure_labels)) - {0})
print("Unique label indices:", unique_r)
print("Tumors in dictionary:", {k: organ_dict_r.get(k) for k in ("GTVp", "GTVn")})
print("Labels present:", [inv_r.get(i, f"idx_{i}") for i in unique_r if i in inv_r])

## 4. Visualize — organs + tumours (PDF per case)

Uses `MedicalImageVisualizer.show_two_niis_side_by_side` with the organ dictionary.
GTVp = red, GTVn = magenta.

In [ ]:
viz = MedicalImageVisualizer()

hecktor_pdf = NOTEBOOK_DIR / f"{HECKTOR_CASE_ID}_test4_separate.pdf"
viz.show_two_niis_side_by_side(
    hecktor_result["image_path"],
    hecktor_result["label_path"],
    save_pdf_path=str(hecktor_pdf),
    show=True,
    organ_dictionary_path=str(ORGAN_DICT_PATH),
)
print("Saved:", hecktor_pdf)

In [ ]:
radcure_pdf = NOTEBOOK_DIR / f"{RADCURE_CASE_ID}_test4_separate.pdf"
viz.show_two_niis_side_by_side(
    radcure_result["image_path"],
    radcure_result["label_path"],
    save_pdf_path=str(radcure_pdf),
    show=True,
    organ_dictionary_path=str(radcure_dict_path),
)
print("Saved:", radcure_pdf)

## 5. Validation checklist

- [ ] HECKTOR: both `GTVp` and `GTVn` appear in the dictionary and on distinct slices
- [ ] RADCURE: at least `GTVp`; `GTVn` if present in RTSTRUCT
- [ ] PDF overlays show red (GTVp) and magenta (GTVn) separately
- [ ] No spurious merged tumour blob where only GTVn exists

When satisfied → **Phase 2**: batch reprocess all cases with `tumor_label_mode="separate"`.